In [33]:
%load_ext autoreload
%autoreload 2

import sys

sys.path.insert(0,"..")
from src.vocab.filtering import is_plausible_number_token, is_plausible_string_token, is_plausible_boolean_token, build_plausible_vocab, is_plausible_token_name
from llm_sdk import Small_LLM_Model
from src.vocab.loader import loader_vocab
from src.fsm.value_matcher import allowed_token_ids_for_number, generate_boolean_value, generate_number_value
from src.generation.decoding import mask_logits, select_next_token, decode_vocab
from src.fsm.value_matcher import generate_string_value
from src.fsm.name_matcher import name_matcher
from src.models.functiondef import FunctionCatalog, FunctionDefinition, Parameter
from src.models.load_init import load_function_catalog, what_function_are_calling

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [34]:
def build_function_definition(caltalog: FunctionCatalog) -> str:
    """Build a string representation of the function definitions in the catalog."""
    description = [f"- {f.name}: {f.description}" for f in catalog.functions]
    return "\n".join(description)

In [35]:
catalog = load_function_catalog("../data/input/functions_definition.json")
descriptions = build_function_definition(catalog)
initial_context = f"Available functions:\n{descriptions}\n\n"

In [36]:
prompt = "Greet shrek"
full_text_json = f'{{ "prompt": "{prompt}", "name": "'
full_text_context = initial_context + full_text_json
model = Small_LLM_Model()
dico_inverse = loader_vocab(model)
dico_decode = decode_vocab(dico_inverse, model)
allowed_characters = {c for w in catalog.functions for c in w.name} 
plausible_number = build_plausible_vocab(dico_inverse, is_plausible_number_token)
plausible_string = build_plausible_vocab(dico_decode, is_plausible_string_token)
plausible_boolean = build_plausible_vocab(dico_inverse, is_plausible_boolean_token)
plausible_token_name = build_plausible_vocab(dico_inverse, lambda x: is_plausible_token_name(x, allowed_characters))
noms_valide = [w.name for w in catalog.functions]
full_text = name_matcher(full_text_context, noms_valide, plausible_token_name, model)
full_text = full_text[len(initial_context):]

In [37]:
function_name = full_text[len(full_text_json):-1]
functions = what_function_are_calling(function_name, catalog)
print(f"full text: {full_text}")
print(f"Function name: {function_name}")
full_text += ', "parameters": {'

full text: { "prompt": "Greet shrek", "name": "fn_greet"
Function name: fn_greet


In [38]:
for index, (key, value) in enumerate(functions.parameters.items()):
    is_last = index == len(functions.parameters) - 1
    full_text += f'"{key}": '
    if value.type == "number":
        full_text += generate_number_value(full_text, plausible_number, model)
    elif value.type == "string":
        full_text += '"'
        result = generate_string_value(full_text, plausible_string, model) + '"'
        full_text += result
    elif value.type == "boolean":
        full_text += generate_boolean_value(full_text, plausible_boolean, model)
    if not is_last:
        full_text += ", "
    else:
        full_text += "}}"
print(full_text)


{ "prompt": "Greet shrek", "name": "fn_greet", "parameters": {"name": "shrek"}}
